In [10]:
# Cell 0: Config
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as stats
import numpy as np
from linearmodels.panel import PanelOLS

SPREADS_PATH = '../output/results/complete_results_weekly.csv'
DATE_COL     = 'date'
COUNTRY_COL  = 'country'
CDS_COL      = 'cds_spread'
YIELD_COL    = 'yield_market'
HORIZON = 5
RECOVERY = 0.4
HORIZONS     = [1, 2, 4, 8]
EXPORTERS    = ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Colombia',
                'Mexico', 'Brazil', 'Egypt', 'Malaysia', 'Qatar']
CONTROLS     = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
                'South Korea', 'Thailand', 'Turkey']

panel = pd.read_csv(SPREADS_PATH, parse_dates=[DATE_COL])

In [11]:
# Cell 2: Load controls and merge onto panel

# ── Load macro controls ───────────────────────────────────────
macro = pd.read_csv(
    '../data/processed/Macroeconomic_variables/macro_risk_variables.csv',
    sep=',', dayfirst=True, parse_dates=['Date'], index_col='Date'
)
vix = pd.read_csv(
    '../data/processed/Macroeconomic_variables/VIXCLS.csv',
    parse_dates=['Date'], index_col='Date'
)
ovx = pd.read_csv(
    '../data/processed/Macroeconomic_variables/OVXCLS.csv',
    parse_dates=['date'], index_col='date'
)
gpr = pd.read_csv(
    '../data/processed/Macroeconomic_variables/geopolitical_risk_index_daily.csv',
    sep=';', dayfirst=True, parse_dates=['date'], index_col='date',
    decimal=','
)
for col in gpr.columns:
    gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',
                           parse_dates=['date'], index_col='date')
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv',
                           dayfirst=True, parse_dates=['date'], index_col='date')

# ── Build controls on daily index ────────────────────────────
macro['VIX']   = vix['VIXCLS']
macro['OVX']   = ovx['OVXCLS']
macro          = macro.join(gpr[['GPRD']], how='left')
macro['basis'] = oil_prices['Brent'] / oil_futures['Brent_12m']
controls_daily = macro[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].copy()

# ── Load FX rates ─────────────────────────────────────────────
fx_wide = pd.read_csv('../data/processed/CCA_V2/exchange_rates.csv',
                       dayfirst=True, parse_dates=['date'])
fx = fx_wide.melt(id_vars='date', var_name=COUNTRY_COL, value_name='fx_rate')
fx = fx.dropna(subset=['fx_rate'])

fx[COUNTRY_COL] = fx[COUNTRY_COL].replace({'United Arab Emirates': 'UAE (Abu Dhabi)'})


# ── Anchor to panel dates ─────────────────────────────────────
cds_dates = pd.DatetimeIndex(panel[DATE_COL].unique())

# ── Reindex controls to panel dates ──────────────────────────
controls_weekly = controls_daily\
    .reindex(cds_dates, method='nearest',
             tolerance=pd.Timedelta('7 days'))\
    .ffill().bfill()\
    .reset_index()\
    .rename(columns={'Date': DATE_COL, 'index': DATE_COL})
controls_weekly[DATE_COL] = pd.to_datetime(controls_weekly[DATE_COL])

# ── Reindex FX per country to panel dates ────────────────────
fx_weekly = pd.merge_asof(
    panel[[DATE_COL, COUNTRY_COL]].sort_values(DATE_COL),
    fx.sort_values(DATE_COL),
    on=DATE_COL,
    by=COUNTRY_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

fx_weekly

# ── Merge controls onto panel ─────────────────────────────────
panel = pd.merge_asof(
    panel.sort_values(DATE_COL),
    controls_weekly.sort_values(DATE_COL),
    on=DATE_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

panel = panel.merge(
    fx_weekly[[DATE_COL, COUNTRY_COL, 'fx_rate']],
    on=[DATE_COL, COUNTRY_COL],
    how='left'
)

panel = panel.sort_values([COUNTRY_COL, DATE_COL]).reset_index(drop=True)

print("Panel shape:", panel.shape)
print("Columns:", panel.columns.tolist())
print("Nulls:\n", panel.isnull().sum()[panel.isnull().sum() > 0])


Panel shape: (8352, 55)
Columns: ['date', 'country', 'cds_spread', 'risk_free_rate', 'B_f', 'LCL_usd', 'sigma_lcl', 'implied_V_M0', 'implied_sigma_V_M0', 'cca_converged_M0', 'implied_V_M1', 'implied_sigma_V_M1', 'cca_converged_M1', 'convenience_yield', 'implied_V_M2', 'implied_sigma_V_M2', 'cca_converged_M2', 'lambda_annual', 'group', 'exporter', 'DD_M0', 'DD_M1', 'DD_M2', 'PD_M0', 'PD_M1', 'PD_M2', 'spread_rf_M0', 'spread_rf_M1', 'spread_rf_M2', 'put_M0', 'put_M1', 'put_M2', 'risky_debt_M0', 'risky_debt_M1', 'risky_debt_M2', 'RNS_M0', 'RNS_M1', 'RNS_M2', 'yield_rf_M0', 'yield_rns_M0', 'yield_rf_M1', 'yield_rns_M1', 'yield_rf_M2', 'yield_rns_M2', 'yield_market', 'leverage_M0', 'leverage_M1', 'leverage_M2', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis', 'fx_rate']
Nulls:
 implied_V_M0          69
implied_sigma_V_M0    69
DD_M0                 69
PD_M0                 69
spread_rf_M0          69
put_M0                69
risky_debt_M0         69
RNS_M0                69
yield_rf_M0      

/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_25052/879418672.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',


# 0: Correlations

In [12]:
# Correlation of levels: CDS vs DD across models
from scipy.stats import pearsonr, spearmanr

print(f"{'Country':<20} | {'M0 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M1 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M2 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8}")
print("-" * 130)

corrs = {f'{m}_{g}_{t}': [] for m in ['M0','M1','M2'] for g in ['exp','ctl'] for t in ['pear','spear']}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    sub = panel[panel[COUNTRY_COL] == c].copy()
    
    line = f"{c:<20}"
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        tmp = sub[[CDS_COL, dd_col]].dropna()
        tmp = tmp[tmp[CDS_COL] > 0]
        
        if len(tmp) > 10:
            pr, pp = pearsonr(tmp[CDS_COL], tmp[dd_col])
            sr, sp = spearmanr(tmp[CDS_COL], tmp[dd_col])
        else:
            pr, pp, sr, sp = np.nan, np.nan, np.nan, np.nan
        
        corrs[f'{model}_{grp}_pear'].append(pr)
        corrs[f'{model}_{grp}_spear'].append(sr)
        
        p_star = '***' if pp < 0.01 else '**' if pp < 0.05 else '*' if pp < 0.1 else ''
        s_star = '***' if sp < 0.01 else '**' if sp < 0.05 else '*' if sp < 0.1 else ''
        line += f" | {pr:>9.3f}{p_star:<3} {pp:>8.4f} {sr:>9.3f}{s_star:<3} {sp:>8.4f}"
    
    print(line)

print("-" * 130)
for g, label in [('exp', 'Exporters'), ('ctl', 'Controls')]:
    line = f"Mean {label:<15}"
    for m in ['M0', 'M1', 'M2']:
        mp = np.nanmean(corrs[f'{m}_{g}_pear'])
        ms = np.nanmean(corrs[f'{m}_{g}_spear'])
        line += f" | {mp:>10.3f} {'':>8} {ms:>10.3f} {'':>8}"
    print(line)

Country              | M0 Pearson        p   Spearman        p | M1 Pearson        p   Spearman        p | M2 Pearson        p   Spearman        p
----------------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |    -0.111**    0.0165    -0.041      0.3845 |    -0.423***   0.0000    -0.413***   0.0000 |    -0.121***   0.0055    -0.029      0.5015
UAE (Abu Dhabi)      |     0.261***   0.0000     0.154***   0.0004 |    -0.456***   0.0000    -0.413***   0.0000 |    -0.411***   0.0000    -0.335***   0.0000
Colombia             |    -0.248***   0.0000    -0.255***   0.0000 |    -0.163***   0.0002    -0.189***   0.0000 |    -0.261***   0.0000    -0.266***   0.0000
Mexico               |    -0.428***   0.0000    -0.472***   0.0000 |    -0.605***   0.0000    -0.494***   0.0000 |    -0.543***   0.0000    -0.475***   0.0000
Brazil               |    -0.369***   0.0000    -0.325***   0.0000 |    -0.489***   0.

# 1. Inseparability Test: OVX and Futures Basis vs Global Risk Factors

In [13]:
# Use unique dates from panel — one row per date for time series regressions
ts = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)\
          [['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].sort_index()

# ── OVX ~ Global factors (levels) ────────────────────────────
idx   = ts[['OVX', 'VIX', 'DXY', 'UST10Y', 'GPRD']].dropna().index
y_ovx = ts.loc[idx, 'OVX']
X_ovx = sm.add_constant(ts.loc[idx, ['VIX', 'DXY', 'UST10Y', 'GPRD']])
res_ovx = sm.OLS(y_ovx, X_ovx).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("OVX ~ Global Factors (levels)")
print(f"R²: {res_ovx.rsquared:.4f}  |  Adj. R²: {res_ovx.rsquared_adj:.4f}")
print(f"\n{'Variable':<12} {'Beta':>10} {'p-value':>10}")
print("-" * 35)
for var in X_ovx.columns:
    print(f"{var:<12} {res_ovx.params[var]:>10.4f} {res_ovx.pvalues[var]:>10.4f}")

OVX ~ Global Factors (levels)
R²: 0.5903  |  Adj. R²: 0.5872

Variable           Beta    p-value
-----------------------------------
const         -101.0461     0.0000
VIX              1.4614     0.0000
DXY              1.3520     0.0000
UST10Y          -7.6438     0.0000
GPRD             0.0104     0.4554


In [14]:
# Convenience yield ~ Global Factors
cy = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)[['convenience_yield', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']].sort_index().dropna()

y = cy['convenience_yield']
X = sm.add_constant(cy[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']])
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("Convenience Yield ~ Global Factors (levels)")
print(f"R²: {res.rsquared:.4f}  |  Adj. R²: {res.rsquared_adj:.4f}")
print(f"\n{'Variable':<15} {'Beta':>10} {'p-value':>10}")
print("-" * 38)
for var in X.columns:
    print(f"{var:<15} {res.params[var]:>10.4f} {res.pvalues[var]:>10.4f}")

Convenience Yield ~ Global Factors (levels)
R²: 0.5105  |  Adj. R²: 0.5057

Variable              Beta    p-value
--------------------------------------
const              -0.2401     0.0180
VIX                 0.0056     0.0000
OVX                -0.0041     0.0000
DXY                 0.0027     0.0302
UST10Y              0.0208     0.0004
GPRD                0.0002     0.0449


# 2. Regressions

In [20]:
# ============================================================
# Regression 1: ln(CDS) = α + β·ln(V) + ε  — Asset Value (M0 vs M1)
# ============================================================

print("Regression 1: ln(CDS) = α + β·V + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)

r2_v = {'M0_exp': [], 'M1_exp': [], 'M0_ctl': [], 'M1_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"

    for model, v_col in [('M0', 'implied_V_M0'), ('M1', 'implied_V_M1')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, v_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[v_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(df[v_col])
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_v[f'{model}_{grp}'].append(res.rsquared)

    print(line)

print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_ctl']):>7.3f}")

Regression 1: ln(CDS) = α + β·V + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.003***  0.0000   0.505   463 |  -0.003***  0.0000   0.557   522
UAE (Abu Dhabi)      |  -0.002***  0.0000   0.384   520 |  -0.002***  0.0000   0.425   522
Colombia             |   0.003***  0.0020   0.106   522 |   0.002***  0.0081   0.083   522
Mexico               |  -0.002***  0.0000   0.171   522 |  -0.001***  0.0000   0.251   522
Brazil               |  -0.002     0.1777   0.021   522 |  -0.004***  0.0000   0.223   522
Egypt                |   0.006***  0.0000   0.525   522 |   0.005***  0.0000   0.509   522
Malaysia             |  -0.011***  0.0000   0.368   520 |  -0.009***  0.0000   0.420   522
Qatar                |  -0.005***  0.0000   0.578   522 |  -0.004***  0.0000   0.591   522
Chile                |  -0.000     0.6713   0.003   522

In [16]:
# ============================================================
# Regression 2: ln(CDS) = α + β·ln(σ) + ε  — Volatility (M0 vs M2)
# ============================================================
 
print("\n\nRegression 2: ln(CDS) = α + β·ln(σ) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)
 
r2_vol = {'M0_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M2_ctl': []}
 
for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
 
    for model, sig_col in [('M0', 'implied_sigma_V_M0'), ('M2', 'implied_sigma_V_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, sig_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[sig_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[sig_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_vol[f'{model}_{grp}'].append(res.rsquared)
 
    print(line)
 
print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_ctl']):>7.3f}")



Regression 2: ln(CDS) = α + β·ln(σ) + ε

Country              |     M0 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |   0.514***  0.0000   0.370   463 |   0.531***  0.0000   0.403   522
UAE (Abu Dhabi)      |  -0.172     0.1429   0.037   520 |   0.297***  0.0002   0.188   522
Colombia             |   0.614***  0.0012   0.147   522 |   0.634***  0.0008   0.157   522
Mexico               |  -0.120     0.3102   0.013   522 |   0.062     0.6808   0.004   522
Brazil               |   0.568***  0.0000   0.232   522 |   0.651***  0.0000   0.285   522
Egypt                |   0.189*    0.0655   0.036   522 |   0.189*    0.0655   0.036   522
Malaysia             |   0.561**   0.0391   0.062   520 |   0.543***  0.0000   0.169   522
Qatar                |   0.537***  0.0000   0.476   522 |   0.571***  0.0000   0.526   522
Chile                |   0.093     0.3553   0.010

In [17]:
print("Regression: ln(CDS) = α + β·DD + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)
r2 = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col]].dropna()
        df = df[df[CDS_COL] > 0]
        y = np.log(df[CDS_COL].values)
        X = sm.add_constant(df[dd_col].values)
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2[f'{model}_{grp}'].append(res.rsquared)
    
    print(line)

print("-" * 120)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2['M1_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2['M1_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2['M2_ctl']):>7.3f}")

Regression: ln(CDS) = α + β·DD + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.029     0.3535   0.010   463 |  -0.081***  0.0000   0.186   522 |  -0.026     0.3698   0.009   522
UAE (Abu Dhabi)      |   0.052*    0.0822   0.061   520 |  -0.029***  0.0009   0.171   522 |  -0.101***  0.0013   0.147   522
Colombia             |  -0.208**   0.0415   0.058   522 |  -0.070     0.1003   0.034   522 |  -0.216**   0.0312   0.064   522
Mexico               |  -0.200***  0.0004   0.195   522 |  -0.108***  0.0000   0.342   522 |  -0.213***  0.0000   0.277   522
Brazil               |  -0.207***  0.0003   0.144   522 |  -0.137***  0.0000   0.237   522 |  -0.221***  0.0001   0.167   522
Egypt                |  -0.125***  0.0003   0.123   522 |  -0.122***  0.0007   0.112   522

In [18]:
print("Regression: ln(CDS) = α + β·DD + ln(VIX) + ln(UST10Y) + ln(DXY) + ln(GPRD) + ln(FX) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)
r2_c = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col, 'VIX', 'UST10Y', 'DXY', 'GPRD', 'fx_rate']].dropna()
        df = df[(df[CDS_COL] > 0) & (df['VIX'] > 0) & (df['UST10Y'] > 0) & (df['DXY'] > 0) & (df['GPRD'] > 0) & (df['fx_rate'] > 0)].reset_index(drop=True)
        
        if len(df) < 30:
            line += f" | {'--':>7}    {'--':>7} {'--':>7} {len(df):>5}"
            continue
        
        y = np.log(df[CDS_COL].values)
        X = np.column_stack([
            np.ones(len(df)),
            df[dd_col].values,
            np.log(df['VIX'].values),
            np.log(df['UST10Y'].values),
            np.log(df['DXY'].values),
            np.log(df['GPRD'].values),
            np.log(df['fx_rate'].values)
        ])
        
        if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
            line += f" | {'--':>7}    {'--':>7} {'--':>7} {len(df):>5}"
            continue
        
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_c[f'{model}_{grp}'].append(res.rsquared)
    
    print(line)

print("-" * 120)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_c['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M1_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_c['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M1_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M2_ctl']):>7.3f}")

Regression: ln(CDS) = α + β·DD + ln(VIX) + ln(UST10Y) + ln(DXY) + ln(GPRD) + ln(FX) + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.067*    0.0560   0.201   463 |  -0.075***  0.0001   0.309   522 |  -0.044     0.1227   0.188   522
UAE (Abu Dhabi)      |   0.039     0.2246   0.133   520 |  -0.030***  0.0000   0.255   522 |  -0.130***  0.0003   0.222   522
Colombia             |  -0.284***  0.0000   0.564   522 |  -0.194***  0.0000   0.637   522 |  -0.292***  0.0000   0.567   522
Mexico               |  -0.287***  0.0000   0.465   522 |  -0.156***  0.0000   0.641   522 |  -0.321***  0.0000   0.545   522
Brazil               |  -0.367***  0.0000   0.399   522 |  -0.210***  0.0002   0.426   522 |  -0.369***  0.0000   0.392   522
Egypt                |  -0.028     0.